# Experimento 4A: PubMedBERT + neg_ratio 1:1

**Objetivo:** aislar el efecto de bajar la proporcion de negativos
(`no_relation`) en train, de 3:1 (baseline) a 1:1, manteniendo todo lo demas
igual. Se construye **sobre el baseline ya arreglado (3A)**, no sobre el
baseline original con bugs -- `fix_entity_markers()` sigue aplicado, es la
base mas correcta disponible ahora mismo (ver `HALLAZGOS-BUGS-TOKENIZACION.md`).

**Como se hace:** `eng_train.txt` ya viene submuestreado a neg_ratio=3 (3193
positivos, 9546 `no_relation`). Para bajar a 1:1 no hace falta volver a los
datos crudos -- basta con quedarse con todos los positivos y submuestrear los
negativos ya existentes a 3193 (mismo numero que positivos), con seed fija
para que sea reproducible. El dev NO se toca (se evalua siempre sobre la
distribucion real).

La comparacion se hace contra **3A** (PubMedBERT + bugs arreglados, sin
typed markers, neg_ratio=3): argmax=0.3338, calibrado=0.4298 @ threshold=0.996
-- es el punto de referencia correcto ahora que el pipeline base esta
arreglado.

## 1. Setup

In [1]:
# Ejecucion en servidor local (zape), entorno conda "tfg". Mismo patron que 1G/2A/2B/3A.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])


HF cache: /home/lucia.esperon/hf_cache/hub


In [2]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_entity_markers
add_macro_f1_metric()
fix_entity_markers()   # <-- los dos fixes de HALLAZGOS-BUGS-TOKENIZACION.md, igual que 3A
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")


/home/lucia.esperon/miniconda3/envs/tfg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA GeForce RTX 2080 Ti (11.5 GB)


## 2. Configuracion

In [3]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_negratio1"
TECHNIQUE       = "neg_ratio 1:1 (submuestreado desde neg_ratio=3) -- sobre fix_entity_markers(), unico cambio vs 3A"
NEG_RATIO_TARGET = 1
DATA_SEED = 42   # seed del submuestreo de negativos (separada del seed de entreno, aunque coincide en valor)

MAX_LENGTH     = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0

DATA_DIR    = Path("../data/english")
DEV_DATA    = DATA_DIR / "eng_dev.txt"          # dev NUNCA se resamplea
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (DATA_DIR / "eng_train.txt", DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/4A-pubmedbert-negratio1/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)


Clases: 15 | modelo: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Salida: ../outputs/4A-pubmedbert-negratio1/seed42


## 3. Submuestrear train a neg_ratio 1:1

In [4]:
train_full = [json.loads(l) for l in open(DATA_DIR / "eng_train.txt", encoding="utf-8") if l.strip()]
positives = [i for i in train_full if i["relation"] != "no_relation"]
negatives = [i for i in train_full if i["relation"] == "no_relation"]
print(f"train original (neg_ratio=3): {len(positives)} positivos, {len(negatives)} negativos, "
      f"ratio={len(negatives)/len(positives):.2f}")

rng = random.Random(DATA_SEED)
n_keep = min(len(positives) * NEG_RATIO_TARGET, len(negatives))
negatives_kept = rng.sample(negatives, n_keep)
train_resampled = positives + negatives_kept
rng.shuffle(train_resampled)

print(f"train submuestreado (neg_ratio={NEG_RATIO_TARGET}): {len(positives)} positivos, "
      f"{len(negatives_kept)} negativos, ratio={len(negatives_kept)/len(positives):.2f}, "
      f"total={len(train_resampled)} (antes {len(train_full)})")

TRAIN_DATA = OUT_DIR / "eng_train_negratio1.txt"
with open(TRAIN_DATA, "w", encoding="utf-8") as f:
    for inst in train_resampled:
        f.write(json.dumps(inst, ensure_ascii=False) + "\n")
print("Guardado:", TRAIN_DATA)


train original (neg_ratio=3): 3193 positivos, 9546 negativos, ratio=2.99
train submuestreado (neg_ratio=1): 3193 positivos, 3193 negativos, ratio=1.00, total=6386 (antes 12739)
Guardado: ../outputs/4A-pubmedbert-negratio1/seed42/eng_train_negratio1.txt


## 4. Entrenamiento -- misma funcion que 1G/2A/2B/3A (`train_with_history`)

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history


In [6]:
set_seed(SEED)

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev)={macro_f1_curado:.4f}")


2026-07-29 14:59:58,830 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-07-29 14:59:58,863 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/config.json "HTTP/1.1 200 OK"


2026-07-29 14:59:59,015 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


2026-07-29 14:59:59,159 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


2026-07-29 14:59:59,161 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


...

 99%|█████████▉| 185/186 [00:15<00:00, 12.13it/s, acc=0.787]

 99%|█████████▉| 185/186 [00:15<00:00, 12.13it/s, acc=0.786]

100%|██████████| 186/186 [00:15<00:00, 12.28it/s, acc=0.786]


2026-07-29 15:29:59,168 - root - INFO - Evaluation result: {'acc': 0.7863161442534546, 'micro_p': 0.8353025420694593, 'micro_r': 0.7863161442534546, 'micro_f1': 0.8100694444444444}.


Epoch 14: loss=0.0096 val_micro_f1=0.8101 val_macro_f1=0.7614
Mejor macro_f1 en val: 0.7668

Entreno: 30.0 min | mejor epoch=8 macro_f1_curado(dev)=0.7668


## 5. Inferencia en blind + evaluacion oficial (argmax)

In [7]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_raw), 64):
        batch = blind_raw[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")


candidatos blind: 282364 | gold real: 2891


Inferencia blind: 24.3 min


Macro F1 ciego (argmax puro): 0.2204


## 6. Calibracion de threshold (grid fino) -- comparacion justa contra 3A

In [8]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def f1_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)["macro_f1"]

FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, f1_at(all_probs, th)) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

# --- referencia: 3A (PubMedBERT + bugs arreglados, neg_ratio=3, sin typed markers) ---
BASELINE_ARGMAX = 0.333788238025547        # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_CALIBRADO = 0.42979245116674064   # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_TH = 0.996

print(f"\n{'':<28}{'argmax':>10}{'calibrado':>12}{'threshold':>12}")
print(f"{'3A (bugs arreglados, neg=3)':<28}{BASELINE_ARGMAX:>10.4f}{BASELINE_CALIBRADO:>12.4f}{BASELINE_TH:>12.3f}")
print(f"{'+ neg_ratio 1:1':<28}{macro_f1_ciego_argmax:>10.4f}{macro_f1_ciego_calibrado:>12.4f}{best_threshold:>12.3f}")
print(f"{'delta':<28}{macro_f1_ciego_argmax-BASELINE_ARGMAX:>+10.4f}{macro_f1_ciego_calibrado-BASELINE_CALIBRADO:>+12.4f}")


Mejor threshold (fino): 0.9950 -> Macro F1 ciego calibrado = 0.3480

                                argmax   calibrado   threshold
3A (bugs arreglados, neg=3)     0.3338      0.4298       0.996
+ neg_ratio 1:1                 0.2204      0.3480       0.995
delta                          -0.1134     -0.0818


## 7. Guardar resultados

In [9]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": NEG_RATIO_TARGET, "seed": SEED,
    },
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "baseline_comparison": {
        "source": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (bugs arreglados, neg_ratio=3)",
        "baseline_macro_f1_ciego_argmax": BASELINE_ARGMAX,
        "baseline_macro_f1_ciego_calibrado": BASELINE_CALIBRADO,
        "delta_argmax": macro_f1_ciego_argmax - BASELINE_ARGMAX,
        "delta_calibrado": macro_f1_ciego_calibrado - BASELINE_CALIBRADO,
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
print(json.dumps(results, indent=2, ensure_ascii=False))


Guardado: ../outputs/4A-pubmedbert-negratio1/seed42/results_seed_summary.json
{
  "exp": "pubmedbert_negratio1",
  "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "technique": "neg_ratio 1:1 (submuestreado desde neg_ratio=3) -- sobre fix_entity_markers(), unico cambio vs 3A",
  "seed": 42,
  "hyperparameters": {
    "max_length": 256,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 15,
    "warmup_steps": 300,
    "neg_ratio": 1,
    "seed": 42
  },
  "macro_f1_curado": 0.7668302852434362,
  "macro_f1_ciego_argmax": 0.22041114524524114,
  "best_threshold_fino": 0.995,
  "macro_f1_ciego_calibrado": 0.3479793039992802,
  "en_borde_del_grid_fino": false,
  "train_minutes": 30.0,
  "blind_minutes": 24.3,
  "baseline_comparison": {
    "source": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (bugs arreglados, neg_ratio=3)",
    "baseline_macro_f1_ciego_argmax": 0.333788238025547,
    "baseline_macro_f1_ciego_calibrado": 0.4297924511

## 8. Conclusion

**Resultado: neg_ratio 1:1 es claramente PEOR que 3:1** (-0.1134 argmax,
-0.0818 calibrado respecto a 3A) -- a diferencia del delta minusculo de 3A
(bugs arreglados) o de typed markers (2A/2B), este delta es grande, muy por
encima del ruido de seed a seed (std~0.006-0.012), asi que es concluyente
incluso con una sola seed.

**Por que probablemente pasa:** el pool real de candidatos en blind es
~98% `no_relation` (282364 candidatos, 2891 relaciones reales). Entrenar con
menos negativos (1:1 en vez de 3:1) hace que el modelo vea proporcionalmente
menos ejemplos de "esto NO es una relacion" durante el entreno, y se vuelve
mas propenso a predecir relaciones falsas en el pool real, mucho mas
desbalanceado que su propio train. El neg_ratio=3 actual ya va en la
direccion correcta (mas negativos, no menos); subir el neg_ratio en vez de
bajarlo (si el pool de negativos disponible lo permite) seria la hipotesis
razonable a probar despues, no bajarlo mas.